# PROC COMPUTAB을 활용한 분기별 예상 손익계산서


## 요약

이 노트북은 SAS/ETS의 보고서 작성용 표 프로시저인 PROC COMPUTAB을 사용하여 월별 원장 데이터로부터 지역 은행의 분기별 예상 손익계산서를 직접 작성합니다. 각 월의 이자수익, 이자비용, 수수료수익, 영업비용을 올바른 역년 분기 열로 배정한 뒤, 행 프로그래밍 블록을 사용하여 순이자수익, 세전이익, 순이익을 계산하고, 열 블록을 사용하여 네 개 분기를 연간 합계로 집계합니다.

## 데이터 출처

단일 합성 데이터셋 `bank_ledger`는 중견 지역사회 은행의 한 회계연도치 월별 재무제표 항목을 시뮬레이션합니다. 열두 개의 월별 관측치가 `call streaminit`/`rand`로 인라인 생성되므로 노트북은 완전히 독립적으로 실행됩니다.

| 변수 | 유형 | 설명 |
|----------|------|-------------|
| `stmtdate` | 숫자 (DATE9.) | 월말 재무제표 일자 (1월~12월) |
| `loanint`  | 숫자 | 대출 포트폴리오에서 벌어들인 이자 및 수수료 (USD 천 단위) |
| `secint`   | 숫자 | 투자증권 계정에서 벌어들인 이자 (USD 천 단위) |
| `depint`   | 숫자 | 예금에 지급한 이자 (USD 천 단위) |
| `borrint`  | 숫자 | 차입금 / FHLB 대출에 지급한 이자 (USD 천 단위) |
| `feeinc`   | 숫자 | 비이자(수수료 및 서비스 청구) 수익 (USD 천 단위) |
| `salaries` | 숫자 | 급여 및 직원 복리후생 비용 (USD 천 단위) |
| `occupancy`| 숫자 | 점포 및 장비 비용 (USD 천 단위) |
| `othropex` | 숫자 | 기타 비이자 영업비용 (USD 천 단위) |
| `provision`| 숫자 | 대손충당금 전입액 (USD 천 단위) |
| `taxrate`  | 숫자 | 세전이익에 적용되는 유효세율 |

# PROC COMPUTAB을 활용한 분기별 예상 손익계산서

은행 재무팀은 일상적으로 월별 총계정원장을 순이자수익과 최종 순이익을 보여주는 **분기별 손익계산서**로 변환합니다. `PROC COMPUTAB`(SAS/ETS)은 바로 이 목적에 특화되어 있습니다. 이 프로시저는 **열**이 보고 기간이고 **행**이 재무제표 항목인 프로그래밍 가능한 표를 배치하며, 행 블록과 열 블록에서 일반적인 DATA 스텝 로직으로 소계를 계산할 수 있게 합니다.

이 노트북에서는 다음을 수행합니다:

1. 지역사회 은행을 위한 한 회계연도치 합성 월별 원장 데이터를 생성합니다.
2. 각 월을 해당 역년 분기로 배정하여 분기별 손익계산서를 작성합니다.
3. **행 블록**에서 순이자수익, 세전이익, 순이익을 계산하고, **열 블록**에서 분기들을 연간 합계로 집계합니다.
4. `out=` 표를 재사용하여 계산된 재무제표가 후속 보고에 활용될 수 있게 합니다.

## 1단계 — 합성 월별 원장 데이터 생성

우리는 열두 개의 월말 관측치를 시뮬레이션합니다. 대출 및 증권 이자수익은 연중 완만하게 상승하고, 예금 및 차입 비용은 금리 환경에 따라 변동하며, 수수료수익, 영업비용, 대손충당금은 현실적인 월별 변동성을 갖습니다. `call streaminit`으로 시드를 고정하여 데이터가 재현 가능합니다.

In [1]:
데이터 bank_ledger;
   호출 streaminit(20240115);
   형식 stmtdate date9.;
   반복 month = 1 까지 12;
      /* 2024 회계연도의 월말 재무제표 일자 */
      stmtdate = mdy(month, 28, 2024);

      /* 연중 완만한 상승 추세 + 월별 변동성 */
      drift = 1 + 0.012 * (month - 1);

      /* 이자수익 (USD 천 단위) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* 이자비용 (USD 천 단위) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* 비이자 수익 및 비용 (USD 천 단위) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* 대손충당금 전입액, 때때로 증가 */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* 유효세율 */
      taxrate = 0.21;

      출력;
   종료;
   유지 stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
실행;

처리 인쇄 데이터=bank_ledger noobs 라벨;
   라벨 stmtdate='결산일'
         loanint='대출이자수익'
         secint='유가증권이자'
         depint='예금이자'
         borrint='차입금이자'
         feeinc='수수료수익'
         salaries='급여'
         occupancy='점포비'
         othropex='기타영업비용'
         provision='대손충당금'
         taxrate='유효세율';
   제목 '합성 월별 원장 — 커뮤니티 은행 2024회계연도';
   형식 loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
실행;

                                              합성 월별 원장 — 커뮤니티 은행 2024회계연도                                               

      결산일              대출이자수익              유가증권이자          예금이자            차입금이자            수수료수익      급여        점포비              기타영업비용            대손충당금          유효세율
28JAN2024            1,772.95              423.79        531.47           128.99           339.33  699.38     171.95              202.43           130.93          0.21
28FEB2024            1,824.38              421.13        564.85           125.53           294.09  767.29     162.79              307.61           123.25          0.21
28MAR2024            1,931.98              442.06        536.64           131.71           295.72  724.03     153.26              254.21           115.76          0.21
28APR2024            1,934.99              439.29        535.94           140.01           294.56  729.47     174.19              237.43           198.05          0.21
28MAY2024            1,949.31         


NOTE: DATA bank_ledger


NOTE: Wrote bank_ledger (12 rows, 11 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=bank_ledger

NOTE: PROC PRINT completed: 12 observations printed, 11 variables


## 2단계 — 분기별 손익계산서 작성

보고서의 핵심은 아래의 `PROC COMPUTAB` 스텝입니다.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`**는 네 개의 분기 열과 연간 열을 정의합니다.
- 레이블이 없는 **입력 블록**은 자동 변수 **`_col_`**을 `qtr(stmtdate)`로 설정하여 각 월별 관측치를 올바른 분기 열로 배정합니다. 입력은 기본적으로 전치되므로, 각 원장 변수는 자신의 이름과 같은 행에 놓입니다.
- **행 블록** `inc_stmt:`는 열마다 한 번씩 실행되어 파생 항목 — 순이자수익, 총 비이자비용, 세전이익, 순이익 — 을 회계사가 하듯이 계산합니다.
- **열 블록** `total:`은 행마다 한 번씩 실행되어 네 개 분기를 `fy`(연간) 열로 합산합니다.

`rows ... / ...` 문은 제목, 들여쓰기, 괘선(`ol` 상단선, `ul` 하단선, `dul` 이중 하단선)을 추가하여 출력이 실제 재무제표처럼 읽히도록 합니다.

In [2]:
제목 '예상 손익계산서 — Community Bancorp, Inc.';
title2 '2024 회계연도  (단위: 천 USD)';

처리 computab 데이터=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* 네 개 분기와 집계된 연간 열 */
   columns qtr1 qtr2 qtr3 qtr4 fy / 형식=comma11.1;
   columns qtr1 / '1분기';
   columns qtr2 / '2분기';
   columns qtr3 / '3분기';
   columns qtr4 / '4분기';
   columns fy   / '연간' '합계' +3;

   /* 손익계산서 행, 위에서 아래로 */
   rows loanint  / '대출 이자 및 수수료';
   rows secint   / '유가증권 이자'              ul;
   rows intinc   / '총 이자수익';
   rows depint   / '예금 이자';
   rows borrint  / '차입금 이자'                ul;
   rows intexp   / '총 이자비용';
   rows nii      / '순이자수익'                 ol skip;
   rows provision/ '대손충당금 전입액'          ul;
   rows niiap    / '충당금 차감 후 순이자수익'  skip;
   rows feeinc   / '비이자수익'                 ul;
   rows salaries / '급여 및 복리후생비';
   rows occupancy/ '점포 및 장비비';
   rows othropex / '기타 영업비용'              ul;
   rows nonintexp/ '총 비이자비용'              skip;
   rows pretax   / '세전이익'                   ol;
   rows taxexp   / '법인세 비용'                ul;
   rows netinc   / '당기순이익'                 dul;

   /* 입력 블록: 각 월을 해당 역년 분기로 배정 */
   _col_ = qtr(stmtdate);

   /* 행 블록: 모든 열에 대해 재무제표 소계를 계산 */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* 열 블록: 네 개 분기를 연간 열로 합산 */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
실행;

                                           예상 손익계산서 — Community Bancorp, Inc.                                           
                                                 2024 회계연도  (단위: 천 USD)                                                 


                             The COMPUTAB Procedure                             

                            1분기          2분기          3분기          4분기           연간  
                                                                                 합계  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  대출 이자 및 수수료
  loanint               5,529.3      5,818.7      5,963.5      6,277.0     23,588.4  
  유가증권 이자
  secint                1,287.0      1,334.9      1,342.0      1,452.9      5,416.8  
                    -----------  -----------  -----------  -----------  -----------  
  총 이자수익
  intinc                6,816.3      7,


NOTE: Option TITLE changed to 예상 손익계산서 — Community Bancorp, Inc..
NOTE: Option TITLE2 changed to 2024 회계연도  (단위: 천 USD).
NOTE: PROC COMPUTAB
NOTE: COMPUTAB OUT= dataset written to: qtr_income.csv
NOTE: PROC COMPUTAB statement used.


## 3단계 — COMPUTAB 출력 데이터셋 재사용

위의 `PROC COMPUTAB` 스텝은 `out=`을 통해 계산된 표를 `qtr_income`에 기록했습니다. 그 데이터셋의 각 행은 재무제표 항목이고 각 열 변수(`qtr1`~`qtr4`, `fy`)는 계산된 값을 담고 있으므로, 후속 보고에 활용될 수 있습니다. 아래에서는 집계된 연간 열을 인쇄하여 수치가 일치하는지 확인합니다.

In [3]:
제목 'COMPUTAB 출력 데이터셋 — 연간 열';

처리 인쇄 데이터=qtr_income 라벨 noobs;
   변수 _row_ fy;
   형식 fy comma12.1;
   라벨 _row_ = '항목' fy = '연간';
실행;

제목;

                                                COMPUTAB 출력 데이터셋 — 연간 열                                                 
                                                 2024 회계연도  (단위: 천 USD)                                                 


       항목        연간
---------  --------
loanint    23,588.4
secint      5,416.8
intinc     29,005.2
depint      6,831.2
borrint     1,650.7
intexp      8,482.0
nii        20,523.2
provision   1,568.9
niiap      18,954.3
feeinc      3,703.2
salaries    8,763.1
occupancy   1,985.6
othropex    2,944.8
nonintexp  13,693.5
pretax      8,964.1
taxexp      1,882.5
netinc      7,081.6




NOTE: Option TITLE changed to COMPUTAB 출력 데이터셋 — 연간 열.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## 결과 해석

예상 재무제표는 은행의 규제용 콜 리포트처럼 위에서 아래로 읽힙니다. 총 이자수익에서 총 이자비용을 뺀 값이 **순이자수익** — 여기서는 연간 약 \$20.5백만 — 으로, 은행의 주된 수익 동력입니다. **대손충당금 전입액**을 빼고, **수수료수익**을 더하며, **비이자비용**을 차감하면 세전이익이 약 \$9.0백만이 되고, 21%의 유효세율을 적용하면 **순이익**이 약 \$7.1백만이 됩니다. `total:` 열 블록은 네 개 분기를 연간 열로 합산하므로, 연간 합계는 구성상 분기들의 합과 일치합니다 — 이는 `out=` 데이터셋에서 확인되며, 연간 `netinc` 값 7,081.6은 네 개 분기 수치의 합과 같습니다.

모든 계산이 `PROC COMPUTAB`의 프로그래밍 가능한 행 블록과 열 블록 안에서 이루어지므로, 실제 월별 원장을 교체하더라도 보고서 로직은 전혀 변경할 필요가 없으며 — 입력 데이터셋만 바뀝니다. 그러면 `out=` 데이터셋은 계산된 재무제표를 대시보드, 추세 분석, 또는 이사회 자료 자동화에 활용할 수 있게 합니다.